# Suite EXP — Expressions

`Expressions.OfLiteral`, `OfOperation`, `OfVariable`, `OfLet` and `OfAny` follow the builder pattern (`Data`,
`Builder`, `Spec`, `resolve`) and have meta-schemas (`Schema`): ordinary object schemas tagged by `kind`, registered as
`Expressions.*`, through which expressions serialize, validate and compare like other objects. Builders implement
`Visitors.OfObject`, which is how `FromPlain(Expressions.Builders)` rebuilds `Data`. `Term`s write expressions with
methods. `Evaluators` computes them (suite EVL).

In [ ]:
import math

from mbse_schemas.Framework import Comparison as C, Evaluators as V, Expressions as E, JSON, Plain, Proxies, Schemas as S, Validators, YAML
from support import Fake, get_property, raises, set_property, text

lit = E.OfLiteral.Builder().value(3).create()
op = E.OfOperation.Builder().name("f").arguments(1, lambda a: a.as_literal("x")).arguments(lit).create()

## EXP-01 · Literals hold one native value

In [ ]:
assert lit.value == 3 and lit.validate() == [] and lit.schema_name() == "Expressions.OfLiteral"
assert E.OfLiteral.resolve(3).value == 3 and E.OfLiteral.resolve(lit) is lit
assert E.OfLiteral.resolve(lambda b: b.value("x")).value == "x"
for value in [0, 2**70, -0.0, math.nan, "", b"", False]:  # every native type
    assert E.OfLiteral.resolve(value).validate() == []
assert E.OfLiteral.Data().validate() == ["a literal needs a value"]
assert E.OfLiteral.Data([1]).validate() == ["a literal must hold a native value, got list"]

## EXP-02 · Operations have a name and ordered arguments

In [ ]:
assert op.name == "f" and [a.value for a in op.arguments] == [1, "x", 3] and op.arguments[2] is lit
assert op.validate() == [] and op.schema_name() == "Expressions.OfOperation"
nested = E.OfOperation.resolve(lambda o: o.name("not").arguments(lambda a: a.as_operation(op)))
assert nested.arguments == (op,) and E.OfOperation.resolve(op) is op
assert E.OfOperation.Data().validate() == ["an operation needs a name"]
assert E.OfOperation.Data("").validate() == ["an operation needs a name"]
assert E.OfOperation.Data(3).validate() == ["an operation's name must be a str, got int"]
bad = E.OfOperation.Data("f", (E.OfLiteral.Data(), E.OfOperation.Data(None, (E.OfLiteral.Data([]),)), 7))
assert bad.validate() == [
    "argument 0: a literal needs a value",
    "argument 1: an operation needs a name",
    "argument 1: argument 0: a literal must hold a native value, got list",
    "argument 2: not an expression: 7",
]

## EXP-03 · Cycles are reported, not followed; shared arguments are not cycles

In [ ]:
loop = E.OfOperation.Builder().name("f").create()
E.OfOperation.Builder(loop).arguments(loop).update()
assert loop.validate() == ["argument 0: the expression contains a cycle"]
assert E.OfOperation.Builder().name("g").arguments(op, op).create().validate() == []

## EXP-04 · Builders: create, clone and update; `OfAny` selects a kind; Spec errors

In [ ]:
source = E.OfOperation.Builder().name("f").arguments(1).create()
copy = E.OfOperation.Builder(source).arguments(2).clone()
assert [a.value for a in copy.arguments] == [1, 2] and len(source.arguments) == 1
assert E.OfOperation.Builder(source).name("g").update() is source and source.name == "g"
literal = E.OfLiteral.Builder().value(3).create()
assert E.OfLiteral.Builder(literal).value(4).clone().value == 4 and literal.value == 3
assert E.OfLiteral.Builder(literal).value(5).update() is literal and literal.value == 5
with raises(ValueError, match="create() is only valid without a source instance"):
    E.OfLiteral.Builder(literal).create()
with raises(ValueError, match="clone() is only valid with a source instance"):
    E.OfOperation.Builder().clone()
with raises(ValueError, match="update() is only valid with a source instance"):
    E.OfLiteral.Builder().update()
with raises(TypeError):
    E.OfLiteral.Builder().create(1)
with raises(TypeError, match="expected operation data to build from, got int"):
    E.OfOperation.Builder(3)
with raises(TypeError, match="expected literal data to build from, got str"):
    E.OfLiteral.Builder("x")

assert E.OfAny.Builder().as_literal(1).create().value == 1
assert E.OfAny.resolve(lambda a: a.as_operation(lambda o: o.name("f"))).name == "f"
assert E.OfAny.resolve(2).value == 2 and E.OfAny.resolve(op) is op
with raises(ValueError, match="create() is only valid without a source instance"):
    E.OfAny.Builder(literal).create()
with raises(ValueError, match="clone() is only valid with a source instance"):
    E.OfAny.Builder().clone()
with raises(ValueError, match="update() is only valid with a source instance"):
    E.OfAny.Builder().update()
with raises(ValueError, match="no kind selected"):
    E.OfAny.Builder().create()
cloned = E.OfAny.Builder(literal).clone()
assert cloned is not literal and cloned.value == literal.value
shallow = E.OfAny.Builder(op).clone()
assert shallow is not op and shallow.name == op.name and shallow.arguments == op.arguments
assert E.OfAny.Builder(literal).as_literal(1).clone().value == 1 and literal.value == 5
assert E.OfAny.Builder(literal).as_literal(9).update() is literal and literal.value == 9
target = E.OfOperation.Builder().name("f").create()
assert E.OfAny.Builder(target).as_operation(lambda o: o.name("h").arguments(1)).update() is target
assert target.name == "h" and [a.value for a in target.arguments] == [1]
with raises(TypeError, match="update() cannot change the kind of the source expression"):
    E.OfAny.Builder(literal).as_operation(op).update()

with raises(TypeError, match="a class is not a Spec here, got int"):
    E.OfAny.resolve(int)
with raises(TypeError, match="a Spec callable must return its builder, got None"):
    E.OfAny.resolve(lambda a: None)
with raises(TypeError, match="expected an operation or a callable taking its builder, got 3"):
    E.OfOperation.resolve(3)
with raises(TypeError, match="expected an expression, a native value or a callable taking its builder, got [1]"):
    E.OfAny.resolve([1])
with raises(TypeError, match="expected a native value, a literal or a callable taking its builder, got None"):
    E.OfLiteral.resolve(None)

## EXP-05 · The meta-schemas are ordinary, registered schemas

In [ ]:
kinds = [E.OfLiteral, E.OfOperation, E.OfVariable, E.OfLet]
for schema in [*(k.Schema for k in kinds), E.Arguments, E.OfAny.Schema]:
    assert schema.validate() == []
assert list(E.OfLiteral.Schema.properties) == ["kind", "int", "float", "str", "bool", "bytes"]
for kind in [E.OfOperation, E.OfVariable, E.OfLet]:
    assert list(kind.Schema.properties) == ["kind", "name"]
assert [list(k.Schema.adjacencies) for k in kinds] == [["used_by"], ["arguments", "used_by"], ["used_by"],
                                                       ["arguments", "used_by"]]
assert E.Arguments.links == ("parent", "argument") and list(E.Arguments.properties) == ["index"]
assert E.Arguments.uniques == (frozenset({"argument"}),)  # the parent and index determine the argument
assert [b.type for b in E.OfAny.Schema.branches] == [k.Schema for k in kinds]
assert [k.Data.KIND for k in kinds] == ["literal", "operation", "variable", "let"]
names = ["Expressions.OfLiteral", "Expressions.OfOperation", "Expressions.OfVariable", "Expressions.OfLet"]
assert [Proxies.schema(name) for name in names] == [k.Schema for k in kinds]
assert [E.Builders.schema(name) for name in names] == [k.Schema for k in kinds]
assert [k.Data().schema_name() for k in kinds] == names
assert Proxies.name_of(E.Arguments) == "Expressions.Arguments" and E.Builders.name_of(E.Arguments) == "Expressions.Arguments"
with raises(TypeError, match="'Expressions.Arguments' is a relation; no relation builder is exposed"):
    E.Builders.schema("Expressions.Arguments")
with raises(AttributeError, match="no schema registered as 'Contact'"):
    E.Builders.schema("Contact")
with raises(LookupError, match="schema is not registered"):
    E.Builders.name_of(S.OfObject.Data())

## EXP-06 · Round trips through Plain, JSON and YAML rebuild `Data`

In [ ]:
expr = E.OfOperation.Builder().name("f").arguments(
    lambda a: a.as_operation(lambda o: o.name("eq").arguments(1, 1.5)), "s", b"\x01", True).create()
expected = JSON.ToJSON.Reachable(E.OfOperation.Schema, expr)
for encode, decode in [(Plain.ToPlain, Plain.FromPlain), (JSON.ToJSON, JSON.FromJSON), (YAML.ToYAML, YAML.FromYAML)]:
    back = decode(E.Builders).Reachable(E.OfOperation.Schema, encode.Reachable(E.OfOperation.Schema, expr))
    assert isinstance(back, E.OfOperation.Data) and back is not expr
    assert back.validate() == [] and JSON.ToJSON.Reachable(E.OfOperation.Schema, back) == expected
not_ = E.OfOperation.Builder().name("not").arguments(True).create()
assert JSON.ToJSON.Reachable(E.OfOperation.Schema, not_) == (
    '{"root": "s0", "objects": {"s0": {"kind": "operation", "name": "not", "arguments": [{"argument": {"$ref": "s1", "$schema": '
    '"Expressions.OfLiteral"}, "index": 0}]}, "s1": {"kind": "literal", "bool": true}}}')
alone = JSON.ToJSON.Reachable(E.OfLiteral.Schema, E.OfLiteral.resolve(-0.0))
assert alone == '{"root": "s0", "objects": {"s0": {"kind": "literal", "float": -0.0}}}'
assert math.copysign(1.0, JSON.FromJSON(E.Builders).Reachable(E.OfLiteral.Schema, alone).value) == -1.0
# nothing set: only the tag is written
assert JSON.ToJSON.Reachable(E.OfLiteral.Schema, E.OfLiteral.Data()) == '{"root": "s0", "objects": {"s0": {"kind": "literal"}}}'
assert JSON.ToJSON.Reachable(E.OfOperation.Schema, E.OfOperation.Data()) == (
    '{"root": "s0", "objects": {"s0": {"kind": "operation"}}}')
assert JSON.ToJSON.Reachable(E.OfVariable.Schema, E.OfVariable.Data()) == '{"root": "s0", "objects": {"s0": {"kind": "variable"}}}'
assert JSON.ToJSON.Reachable(E.OfLet.Schema, E.OfLet.Data()) == '{"root": "s0", "objects": {"s0": {"kind": "let"}}}'
with raises(ValueError, match="expected kind 'literal', got 'operation'"):
    JSON.FromJSON(E.Builders).Reachable(E.OfLiteral.Schema, '{"root": "s0", "objects": {"s0": {"kind": "operation"}}}')

## EXP-07 · Proxies build expressions too; the `used_by` entries they add are implied and ignored

In [ ]:
proxy = JSON.FromJSON(Proxies.Builders).Reachable(E.OfOperation.Schema, expected)
with_inverse = JSON.ToJSON.Reachable(E.OfOperation.Schema, proxy)
assert '"used_by"' in with_inverse and with_inverse != expected
back = JSON.FromJSON(E.Builders).Reachable(E.OfOperation.Schema, with_inverse)
assert JSON.ToJSON.Reachable(E.OfOperation.Schema, back) == expected

## EXP-08 · Validation and comparison work through the meta-schemas

In [ ]:
validate = Validators.Validate(E.Builders)
assert validate.Reachable(E.OfOperation.Schema, expr) == [] and validate(E.OfLiteral.Schema, lit) == []
assert validate(E.OfOperation.Schema, E.OfOperation.Data(3)) == ["Expressions.OfOperation#0.name: expected str, got int"]
with raises(TypeError, match="a literal must hold a native value, got list"):
    JSON.ToJSON.Reachable(E.OfLiteral.Schema, E.OfLiteral.Data([1]))


def record(schema, expression):
    return C.OfObject(schema, expression)


one, two = E.OfLiteral.resolve(1), E.OfLiteral.resolve(2)
assert record(E.OfLiteral.Schema, one).compare(record(E.OfLiteral.Schema, E.OfLiteral.resolve(1))) == 0
assert record(E.OfLiteral.Schema, one).compare(record(E.OfLiteral.Schema, two)) is None  # objects are not ordered
assert record(E.OfLiteral.Schema, one).compare(record(E.OfLiteral.Schema, E.OfLiteral.resolve(1.0))) is None
f1 = E.OfOperation.resolve(lambda o: o.name("f").arguments(1))
f2 = E.OfOperation.resolve(lambda o: o.name("f").arguments(2))
assert record(E.OfOperation.Schema, f1).compare(record(E.OfOperation.Schema, f2)) == 0  # arguments are adjacencies

## EXP-09 · Builders read and write through the visitor protocols

In [ ]:
seen = []
b = E.OfLiteral.Builder()
b.properties(lambda p: seen.append(p.name()))
assert seen == ["kind"] and b.has("kind") and get_property(b, "kind") == "literal"  # the tag is always there
set_property(b, "kind", "literal")
assert b.clear("kind") is b and b.has("kind")
with raises(ValueError, match="expected kind 'literal', got 'let'"):
    set_property(b, "kind", "let")
seen.clear()
set_property(b, "float", 1.5)
assert b.has("float") and not b.has("int") and get_property(b, "float") == 1.5
set_property(b, "str", "s")  # another type replaces the value
assert not b.has("float") and b.create().value == "s"
b.properties(lambda p: seen.append((p.name(), p.has())))
assert seen == [("kind", True), ("str", True)]
assert b.clear("int") is b and b.has("str")  # clearing another type changes nothing
assert b.clear("str") is b and b.create().value is None
with raises(KeyError, match="unknown property 'nope'"):
    b.property("nope", lambda p: None)
with raises(TypeError, match="expected int, got bool"):
    set_property(b, "int", True)
with raises(AttributeError, match="property 'int' is not set"):
    get_property(b, "int")
b.property("int", lambda p: p.value(lambda a: a.as_native(lambda n: n.set(1).clear())).clear())
assert not b.has("int")
for method in ["as_object", "as_union", "as_intersection"]:
    with raises(TypeError, match="property 'int' is native"):
        b.property("int", lambda p, m=method: p.value(lambda a: getattr(a, m)(lambda v: None)))
seen.clear()
assert b.adjacencies(lambda a: seen.append((a.name(), a.me()))) is b and seen == [("used_by", "argument")]
with raises(KeyError, match="unknown adjacency 'nope'"):
    b.adjacency("nope", lambda a: None)
seen.clear()
b.adjacency("used_by", lambda a: (a.add(lambda e: e.link("parent", lambda k: k.set(op))), a.entries(seen.append),
                                  a.remove(None)))
assert seen == []  # used_by entries are ignored

o = E.OfOperation.Builder()
set_property(o, "name", "f")
assert o.has("name") and not o.has("nope") and o.clear("nope") is o and get_property(o, "name") == "f"
seen.clear()
o.properties(lambda p: seen.append(p.name()))
assert o.clear("name") is o and not o.has("name")
o.properties(lambda p: seen.append(p.name()))
assert seen == ["kind", "name", "kind"]
with raises(KeyError, match="unknown property 'arity'"):
    o.property("arity", lambda p: None)
with raises(KeyError, match="unknown adjacency 'nope'"):
    o.adjacency("nope", lambda a: None)
seen.clear()
o.adjacencies(lambda a: seen.append((a.name(), a.me())))
assert seen == [("arguments", "parent"), ("used_by", "argument")]


def add(builder, index, target):
    """Adds an `arguments` entry through the protocol, as FromPlain does."""
    def fill(entry):
        if target is not None:
            entry.link("argument", lambda k: k.set(target))
        if index is not None:
            set_property(entry, "index", index)
    builder.adjacency("arguments", lambda a: a.add(fill))


w, x, y, z = (E.OfLiteral.resolve(v) for v in "wxyz")
o.name("f")
add(o, 1, y)
add(o, None, z)
add(o, 0, x)
add(o, None, w)
assert [a.value for a in o.create().arguments] == ["x", "y", "z", "w"]  # by index; entries without one last, in order
seen.clear()


def read(entry):
    entry.links(lambda k: (seen.append(k.name()), k.target(seen.append)))
    entry.properties(lambda p: seen.append(p.name()))
    seen.append(get_property(entry, "index") if entry.has("index") else None)
    seen.append(entry.has("index"))
    entry.clear("other")
    entry.clear("index")
    seen.append(entry.has("index"))


o.adjacency("arguments", lambda a: a.entries(lambda e: (read(e), seen.append("|"))))
assert seen[:7] == ["argument", y, "index", 1, True, False, "|"]
with raises(KeyError, match="'parent' is not a link this entry can set"):
    o.adjacency("arguments", lambda a: a.entries(lambda e: e.link("parent", lambda k: None)))
with raises(KeyError, match="unknown property 'nope'"):
    o.adjacency("arguments", lambda a: a.entries(lambda e: e.property("nope", lambda p: None)))
o.adjacency("arguments", lambda a: a.entries(lambda e: a.remove(e)))
assert o.create().arguments == ()
add(o, 0, None)
with raises(ValueError, match="link 'argument' is not set"):
    o.create()
with raises(ValueError, match="link 'argument' is not set"):
    o.adjacency("arguments", lambda a: a.entries(lambda e: e.link("argument", lambda k: k.target(seen.append))))
fake = E.OfOperation.Builder().name("f")
add(fake, 0, Fake("Contact"))
with raises(TypeError, match="an argument must be an expression, got Fake"):
    fake.create()

## EXP-10 · Variables and lets

In [ ]:
v = E.OfVariable.Builder().name("x").create()
assert v.name == "x" and v.schema_name() == "Expressions.OfVariable" and E.OfVariable.resolve("x").name == "x"
assert v.validate() == ["variable 'x' is not bound"] and v.validate(bound={"x"}) == []
assert E.OfVariable.Data().validate() == ["a variable needs a name"]
assert E.OfVariable.Data(1).validate() == ["a variable's name must be a str, got int"]
assert E.OfVariable.resolve(v) is v and E.OfVariable.Builder(v).name("y").clone().name == "y"
let = E.OfLet.Builder().name("x").value(1).body(v).create()
assert let.name == "x" and let.value.value == 1 and let.body is v and let.validate() == []
assert E.OfLet.Builder(let).body(2).clone().body.value == 2 and let.body is v
assert E.OfLet.Builder(let).value(5).update() is let and let.value.value == 5 and let.body is v
assert E.OfLet.resolve(lambda b: b.name("y").value(1).body(2)).name == "y" and E.OfLet.resolve(let) is let
assert E.OfLet.Data("x").validate() == ["a let needs a value", "a let needs a body"]
assert E.OfLet.Data(None, E.OfLiteral.resolve(1), E.OfVariable.resolve("x")).validate() == [
    "a let needs a name", "body: variable 'x' is not bound"]
same_name = E.OfLet.Data("y", E.OfVariable.resolve("y"), E.OfVariable.resolve("y"))
assert same_name.validate() == ["value: variable 'y' is not bound"]  # the name is bound in the body only
assert E.OfAny.resolve(lambda a: a.as_variable("x")).name == "x" and E.OfAny.resolve(lambda a: a.as_let(let)) is let
assert E.OfAny.Builder(v).clone().name == "x" and E.OfAny.Builder(let).clone().body is v
unnamed = [E.OfOperation.Builder().create(), E.OfVariable.Builder().create(), E.OfLet.Builder().create()]
assert [u.name for u in unnamed] == [None, None, None] and unnamed[2].value is None
open_let = E.OfLet.Builder(E.OfLet.Data("x")).clone()
assert open_let.value is None and open_let.body is None
with raises(TypeError, match="expected a let or a callable taking its builder, got 3"):
    E.OfLet.resolve(3)
with raises(TypeError, match="expected a name, a variable or a callable taking its builder, got 3"):
    E.OfVariable.resolve(3)
vb = E.OfVariable.Builder()
seen = []
assert vb.adjacencies(lambda a: seen.append(a.name())) is vb and seen == ["used_by"]  # a variable has no arguments
with raises(KeyError, match="unknown adjacency 'arguments'"):
    vb.adjacency("arguments", lambda a: None)
bad = E.OfLet.Builder().name("x")
add(bad, 2, E.OfLiteral.resolve(1))
with raises(ValueError, match="a let's value is argument 0 and its body argument 1, got index 2"):
    bad.create()

## EXP-11 · Terms write expressions with methods

In [ ]:
this = E.variable("this")
term = this.age.ge(18)
assert isinstance(term, E.Term) and term.data.name == "ge" and term.data.arguments[1].value == 18
get = term.data.arguments[0]
assert get.name == "get" and get.arguments[0] is this.data and get.arguments[1].value == "age"
assert this.get("eq").data.arguments[1].value == "eq"  # a property named like a method
methods = [this.eq(1), this.ne(1), this.lt(1), this.le(1), this.gt(1), this.ge(1), this.and_(True), this.or_(True),
           this.not_(), this.implies(True), this.add(1), this.sub(1), this.mul(1), this.neg(), this.has("x"),
           this.get("x")]
assert [m.data.name for m in methods] == ["eq", "ne", "lt", "le", "gt", "ge", "and", "or", "not", "implies", "add",
                                          "sub", "mul", "neg", "has", "get"]
assert E.literal(3).data.value == 3 and E.operation("in", 1, 2).data.name == "in"
bound = E.let_("a", 1, E.variable("a"))
assert bound.data.name == "a" and E.OfLet.resolve(bound) is bound.data and E.OfOperation.resolve(term) is term.data
assert E.OfAny.resolve(term) is term.data and E.OfAny.resolve(lambda a: a.as_operation(term)) is term.data
with raises(TypeError, match="expected an operation or a callable taking its builder"):
    E.OfOperation.resolve(this)  # a variable, not an operation
with raises(AttributeError):
    this._private  # Python's own probes are not properties

## EXP-12 · `from_` reads an expression from a Python function (Python only)

In [ ]:
import linecache


def text_of(expression):
    """The JSON of an expression, for comparing two by their content and sharing."""
    data = E.OfAny.resolve(expression)
    return JSON.ToJSON.Reachable(Proxies.schema(data.schema_name()), data)


def same_as(function, term):
    assert text_of(E.from_(function)) == text_of(term), (text_of(E.from_(function)), text_of(term))


MIN = 18
this, t, a, one = E.variable("this"), E.variable("t"), E.variable("a"), E.literal(1)
same_as(lambda this: this.age >= MIN and this.email is not None, this.age.ge(18).and_(this.has("email")))
same_as(lambda t: t.a < 1 < t.b, t.a.lt(one).and_(one.lt(t.b)))  # a chain; the middle operand is shared
same_as(lambda t: -t.a * 2 + +t.b - 1 == 0, t.a.neg().mul(2).add(t.b).sub(1).eq(0))
same_as(lambda t: (lambda a: a.b > a.c)(t.d), E.let_("a", t.d, a.b.gt(a.c)))  # a let


def rule(t):
    """A def works too, if its body is one return."""
    return not hasattr(t, "x") or getattr(t, "y") != 2 or t.z is None


same_as(rule, t.has("x").not_().or_(t.get("y").ne(2)).or_(t.has("z").not_()))


def plain(t):
    return t.a


same_as(plain, t.a)
first, second = (lambda t: t.a), (lambda t: t.b)  # two on one line
same_as(first, t.a)
same_as(second, t.b)
LIMIT, LIMIT_DATA = E.literal(5), E.OfLiteral.resolve(6)
assert E.from_(lambda: LIMIT).data is LIMIT.data and E.from_(lambda: LIMIT_DATA).data is LIMIT_DATA


def at_least(n):
    return lambda t: t.a >= n  # a closure: n is read when from_ runs


same_as(at_least(3), t.a.ge(3))
assert V.OfAny(E.from_(lambda: 1 == 1.0)) is None  # the expression's rules, not Python's

for function, message in [
    (lambda t: t.a / 2, "cannot convert 't.a / 2': not supported in an expression"),
    (lambda t: t.a if t else 1, "cannot convert 't.a if t else 1': not supported in an expression"),
    (lambda t: ~t.a, "cannot convert '~t.a': not supported in an expression"),
    (lambda t: t.a in t, "cannot convert 't.a in t': not supported in an expression"),
    (lambda t: t.a is not t, "cannot convert 't.a is not t': 'is' only tests whether a property is None"),
    (lambda: None, "cannot convert 'None': only native constants are literals"),
    (lambda: undefined_name, "cannot convert 'undefined_name': the name is not defined"),  # noqa: F821
    (lambda: linecache, "cannot convert 'linecache': a module is not a native value or an expression"),
    (lambda t=1: t, "only plain positional parameters can become variables"),
    (lambda: (lambda a, b: a)(1, 2), "cannot convert '(lambda a, b: a)(1, 2)': a let binds one name"),
    (lambda t: hasattr(t, 1), "cannot convert 'hasattr(t, 1)': hasattr takes an object and a property name"),
    (lambda t: hasattr(t, "x", y=1), """cannot convert "hasattr(t, 'x', y=1)": not supported in an expression"""),
    (lambda t: len(t), "cannot convert 'len(t)': not supported in an expression"),
    (lambda hasattr: hasattr(1, "x"), """cannot convert "hasattr(1, 'x')": not supported in an expression"""),
]:
    with raises(ValueError, match=message):
        E.from_(function)


def two_statements(t):
    t = t
    return t


def bare(t):
    return


for function in [two_statements, bare]:
    with raises(ValueError, match=f"the body of {function.__name__!r} must be a single return statement"):
        E.from_(function)
with raises(TypeError, match="expected a Python function, got builtin_function_or_method"):
    E.from_(len)
with raises(ValueError, match="the function's source is not available"):
    E.from_(eval("lambda: 1"))
linecache.cache["<unparsable>"] = (9, None, ["def (\n"], "<unparsable>")
with raises(ValueError, match="the function's source is not available"):
    E.from_(eval(compile("lambda: 1", "<unparsable>", "eval")))